In [2]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Même chose que précédemment
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    # J'ai attendu un cours de HuggingFace toute ma vie.
    "This course is amazing!",  # Ce cours est incroyable !
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")


/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7266.69it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                     

In [3]:
#pas de label, impossible de calculer de loss
outputs = model(**batch)
print("loss sans labels :", outputs.loss)
print("logits :", outputs.logits)


loss sans labels : None
logits : tensor([[ 0.1926, -0.0489],
        [ 0.2278, -0.0033]], grad_fn=<AddmmBackward0>)


In [4]:
#label , on donne les bonnes rep. au modele
batch["labels"] = torch.tensor([1, 1])
print("clés du batch :", list(batch.keys()))

clés du batch : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [5]:
# L'opt. recoit tous les poids du modle
optimizer = AdamW(model.parameters())
nb_poids = sum(p.numel() for p in model.parameters())
print("nombre de poids :", nb_poids)

nombre de poids : 109483778


In [6]:
# La loss avec label
loss = model(**batch).loss
print("loss :", loss)

loss : tensor(0.8183, grad_fn=<NllLossBackward0>)


In [7]:
# backward calcule les gradients, sans toucher aux poids
print("gradient avant :", model.classifier.weight.grad)
loss.backward()
print("gradient après :", model.classifier.weight.grad.shape)

gradient avant : None
gradient après : torch.Size([2, 768])


In [8]:
poids_avant = model.classifier.weight.detach().clone()
optimizer.step()
poids_apres = model.classifier.weight.detach()

print("5 premiers poids AVANT :", poids_avant[1, :5])
print("5 premiers poids APRÈS :", poids_apres[1, :5])

5 premiers poids AVANT : tensor([-0.0059, -0.0041, -0.0259,  0.0274, -0.0228])
5 premiers poids APRÈS : tensor([-0.0069, -0.0051, -0.0269,  0.0284, -0.0218])


In [9]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [10]:
train = raw_datasets["train"]
print("train: ", train)
print("train element 0: ",train[0])
print("len raw_datasets: ", len( raw_datasets))
print("len train: ", len(train))
print("derniere element: ", train[-1])
print("tranche :2 ", train[:2], '\n'"pair 0: ", train[0])

train:  Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})
train element 0:  {'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': 1, 'idx': 0}
len raw_datasets:  3
len train:  3668
derniere element:  {'sentence1': "The 30-year bond US30YT = RR rose 22 / 32 for a yield of 4.31 percent , versus 4.35 percent at Wednesday 's close .", 'sentence2': 'The 30-year bond US30YT = RR grew 1-3 / 32 for a yield of 4.30 percent , down from 4.35 percent late Wednesday .', 'label': 0, 'idx': 4075}
tranche :2  {'sentence1': ['Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion ."], 'sentence2': ['Referring to him as only "

In [11]:

print(train.features)
valid = raw_datasets["validation"]
print(valid.features)

label_feature = train.features["label"]

print(label_feature)

print(train.features["label"].names)
print(train.features["sentence1"])

print(label_feature.names)
print(label_feature.int2str(0))
print(label_feature.int2str(1))

print("=================\n")

exemple = train[15]
print(exemple)
print(exemple["label"], label_feature.int2str(exemple["label"]))

{'sentence1': Value('string'), 'sentence2': Value('string'), 'label': ClassLabel(names=['not_equivalent', 'equivalent']), 'idx': Value('int32')}
{'sentence1': Value('string'), 'sentence2': Value('string'), 'label': ClassLabel(names=['not_equivalent', 'equivalent']), 'idx': Value('int32')}
ClassLabel(names=['not_equivalent', 'equivalent'])
['not_equivalent', 'equivalent']
Value('string')
['not_equivalent', 'equivalent']
not_equivalent
equivalent

{'sentence1': 'Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .', 'sentence2': 'Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .', 'label': 0, 'idx': 16}
0 not_equivalent


In [12]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
print(inputs)

tokenizer.convert_ids_to_tokens(inputs["input_ids"])

{'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


['[CLS]',
 'this',
 'is',
 'the',
 'first',
 'sentence',
 '.',
 '[SEP]',
 'this',
 'is',
 'the',
 'second',
 'one',
 '.',
 '[SEP]']

In [43]:
sentence_1 = exemple["sentence1"]
sentence_2 = exemple["sentence2"]

exemple_tokenize = tokenizer(sentence_1)
exemple_tokenize2 = tokenizer(sentence_2)
pair_tokenize = tokenizer(sentence_1, sentence_2)

print("tokenizer sentence 1: ", exemple_tokenize)
print("tokenizer sentence 2: ", exemple_tokenize2)
print("pair tokenizer: ", pair_tokenize)

print("\nlen sentence 1: ", len(exemple_tokenize["input_ids"]))
print("len sentence 2: ", len(exemple_tokenize2["input_ids"]))
print("len pair:       ", len(pair_tokenize["input_ids"]))

#tokenizer.convert_ids_to_tokens(exemple_tokenize["input_ids"])


tokenizer.convert_ids_to_tokens(pair_tokenize["input_ids"])


tokenizer sentence 1:  {'input_ids': [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
tokenizer sentence 2:  {'input_ids': [101, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
pair tokenizer:  {'input_ids': [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102], 'tok

['[CLS]',
 'rudder',
 'was',
 'most',
 'recently',
 'senior',
 'vice',
 'president',
 'for',
 'the',
 'developer',
 '&',
 'platform',
 'evan',
 '##gel',
 '##ism',
 'business',
 '.',
 '[SEP]',
 'senior',
 'vice',
 'president',
 'eric',
 'rudder',
 ',',
 'formerly',
 'head',
 'of',
 'the',
 'developer',
 'and',
 'platform',
 'evan',
 '##gel',
 '##ism',
 'unit',
 ',',
 'will',
 'lead',
 'the',
 'new',
 'entity',
 '.',
 '[SEP]']

In [ ]:
tokenized_dataset = tokenizer(
    list(train["sentence1"]),
    list(train["sentence2"]),
    padding=True,
    truncation=True,
)
print("key :", list(tokenized_dataset.keys()))
print("nombre de paires :", len(tokenized_dataset["input_ids"]))
print("longueur paire 0 :", len(tokenized_dataset["input_ids"][0]))
print("longueur paire 1 :", len(tokenized_dataset["input_ids"][1]))
print("fin paire 0, input_ids :", tokenized_dataset["input_ids"][0][-10:])
print("fin paire 0, attention_mask :", tokenized_dataset["attention_mask"][0][-10:])

clés : ['input_ids', 'token_type_ids', 'attention_mask']
nombre de paires : 3668
longueur paire 0 : 103
longueur paire 1 : 103
fin paire 0, input_ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
fin paire 0, attention_mask : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [46]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
print(tokenized_datasets)
print("longueurs des 5 premières paires :", [len(x) for x in tokenized_datasets["train"][:5]["input_ids"]])

Map: 100%|██████████| 1725/1725 [00:00<00:00, 46862.06 examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})
longueurs des 5 premières paires : [50, 59, 47, 67, 59]


In [47]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
print("longueurs avant :", [len(x) for x in samples["input_ids"]])

batch = data_collator(samples)
print({k: v.shape for k, v in batch.items()})

longueurs avant : [50, 59, 47, 67, 59, 50, 62, 32]
{'input_ids': torch.Size([8, 67]), 'token_type_ids': torch.Size([8, 67]), 'attention_mask': torch.Size([8, 67]), 'labels': torch.Size([8])}
